# Training
### Purpose: create train and validation datasets, build augmentations, run Trainer, save versioned checkpoints.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

import torch

from src.data.annotations import load_json_annotations
from src.data.augmentations import get_train_augmentations, get_val_augmentations
from src.data.loaders import ImageMaskDataset
from src.models.zoo import MODEL_BUILDERS
from src.training.engine import run_training
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t, c
from src.models.zoo import build_model
from src.prediction.validation import analyze_validation_metrics
from torch.nn import CrossEntropyLoss
from src.training.running import get_version_config
from src.training.trainer import create_splits

config = Config.load(root = root)

init_notebook(config.train.seed)

train_dir = config.paths.train_images
annotations_path = config.paths.annotations
entries = load_json_annotations(annotations_path)

# Shuffle entries
train_entries, val_entries = create_splits(entries)



auto_adjust disabled
=== init_notebook ===
Done


#### Dataset split

In [2]:
# Build augmentation pipelines for training and validation
train_tf = get_train_augmentations(config.train.image_size)
val_tf = get_val_augmentations(config.train.image_size)

# Build dataset objects that load image-mask pairs and apply transforms
train_ds = ImageMaskDataset(train_entries, train_dir, transform = train_tf)
val_ds = ImageMaskDataset(val_entries, train_dir, transform = val_tf)

# Build dataloaders
train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size = config.train.batch_size,
        shuffle = True,
        num_workers = config.train.num_workers,
)

val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size = config.train.batch_size,
        shuffle = False,
        num_workers = config.train.num_workers,
)

p("Train samples", len(train_ds))
p("Val samples", len(val_ds))


Train samples: 120
Val samples: 30


#### Available Models

In [3]:
p("Models", MODEL_BUILDERS)
#config.show()
p("Batch", config.train.batch_size)
p("Epochs", config.train.epochs)
p("Learning Rate", config.train.learning_rate, precision = 9)
p("Image Size", config.train.image_size)


Models: 12 keys
  simple_cnn: <function create_simple_cnn at 0x162ae9300>
  unet: <function create_unet at 0x162b3c400>
  smp_unet: <function create_smp_unet at 0x162b3d260>
  smp_fpn: <function create_smp_fpn at 0x162b3d300>
  smp_linknet: <function create_smp_linknet at 0x162b3d3a0>
  smp_deeplabv3: <function create_smp_deeplabv3 at 0x162b3d440>
  smp_deeplabv3plus: <function create_smp_deeplabv3plus at 0x162b3d4e0>
  segformer: <function create_segformer at 0x162b3d580>
  yolov8n: <function create_yolov8n at 0x162b3d620>
  yolov8s: <function create_yolov8s at 0x162b3d6c0>
  yolov8m: <function create_yolov8m at 0x162b3d760>
  yolov8l: <function create_yolov8l at 0x162b3d800>
Batch: 3
Epochs: 10
Learning Rate: 0.000400000
Image Size: 64


#### Run Training

In [4]:
# Create structured path: checkpoints/notebook_eval/[model]/[mode]/[version]
model_name, mode, in_channels = "simple_cnn", "rgb", "3"

key, version_root, exp_config, best_model_path, checkpoint_path_check, best_model_exists = get_version_config(
        config, None, "03", model_name, mode, in_channels, None, None, None
)

trainer = run_training(
        config = exp_config,
        train_loader = train_loader,
        val_loader = val_loader,
        version_root = version_root,
        model_name = model_name,
)


Version root: /Volumes/Colibri/CascadeProjects/cap6415-tree-canopy-detection/checkpoints/03/simple_cnn/rgb/size_64
key: simple_cnn_rgb
=== Validating Training ===
Image Batch 0::   Images: torch.Size([3, 3, 64, 64]), dtype=torch.float32, range=[-1.793, 2.466]
Mask Batch 0::   Masks: torch.Size([3, 64, 64]), dtype=torch.int64, unique=[0, 1, 2]
Image Batch 1::   Images: torch.Size([3, 3, 64, 64]), dtype=torch.float32, range=[-1.570, 2.640]
Mask Batch 1::   Masks: torch.Size([3, 64, 64]), dtype=torch.int64, unique=[0, 1, 2]
Image Batch 2::   Images: torch.Size([3, 3, 64, 64]), dtype=torch.float32, range=[-1.844, 2.413]
Mask Batch 2::   Masks: torch.Size([3, 64, 64]), dtype=torch.int64, unique=[0, 1, 2]
=== Validating Validation ===
Image Batch 0::   Images: torch.Size([3, 3, 64, 64]), dtype=torch.float32, range=[-1.947, 2.623]
Mask Batch 0::   Masks: torch.Size([3, 64, 64]), dtype=torch.int64, unique=[0, 1, 2]
Image Batch 1::   Images: torch.Size([3, 3, 64, 64]), dtype=torch.float32, rang

In [5]:
# Verify tensor types
t("Tensor Type Verification")
sample_img, sample_mask = train_ds[0]
p(f"Image dtype: {sample_img.dtype}, shape: {sample_img.shape}")
p(f"Mask dtype: {sample_mask.dtype}, shape: {sample_mask.shape}", color1 = c.BLUE)
p(f"Mask values: min={sample_mask.min()}, max={sample_mask.max()}", color1 = c.BLACK)
p(f"Mask unique values: {torch.unique(sample_mask)}", color1 = c.BLACK)

# Test a batch
batch_imgs, batch_masks = next(iter(train_loader))
p(f"\nBatch image dtype: {batch_imgs.dtype}, shape: {batch_imgs.shape}")
p(f"Batch mask dtype: {batch_masks.dtype}, shape: {batch_masks.shape}", color1 = c.BLUE)

# Test with model
model = build_model('simple_cnn', in_channels = 3, out_channels = 3)
with torch.no_grad():
    preds = model(batch_imgs[:1])
p(f"\nModel output dtype: {preds.dtype}, shape: {preds.shape}", color1 = c.BLACK)

# Test loss
criterion = CrossEntropyLoss()
loss = criterion(preds, batch_masks[:1])
p(f"Loss computed successfully: {loss.item()}", color1 = c.BLACK)


=== Tensor Type Verification ===
Image dtype: torch.float32, shape: torch.Size([3, 64, 64])
Mask dtype: torch.int64, shape: torch.Size([64, 64])
Mask values: min=0, max=1
Mask unique values: tensor([0, 1])

Batch image dtype: torch.float32, shape: torch.Size([3, 3, 64, 64])
Batch mask dtype: torch.int64, shape: torch.Size([3, 64, 64])

Model output dtype: torch.float32, shape: torch.Size([1, 3, 64, 64])
Loss computed successfully: 1.3299386501312256


In [6]:
if trainer is not None:
    checkpoint_path = trainer.paths["checkpoint"]
    if checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location = "cpu")

        analyze_validation_metrics(
                val_loss = ckpt.get("val_loss", 0),
                iou = ckpt.get("val_iou", ckpt.get("iou", 0)),
                accuracy = ckpt.get("val_accuracy", ckpt.get("accuracy", 0)),
                precision = ckpt.get("val_precision", ckpt.get("precision", 0)),
                recall = ckpt.get("val_recall", ckpt.get("recall", 0)),
                f1_score = ckpt.get("val_f1_score", ckpt.get("f1_score", 0)),
                individual_tree_iou = ckpt.get("val_iou_individual", 0),
                group_tree_iou = ckpt.get("val_iou_group", 0),
                dice = ckpt.get("val_dice", ckpt.get("dice", 0)),
                model_name = "simple_cnn"
        )


=== simple_cnn Performance Analysis ===
Performance Categories: 🥇 excellent |🥈 good |🥉 fair | 😡 poor

Overall Performance: POOR
Overall Score: 1.33/4.0

=== Metric Breakdown: ===
  🥉 Val Loss: 0.8709 (fair)
  😡 Iou: 24.3% (poor)
  😡 Accuracy: 59.3% (poor)
  😡 Precision: 33.8% (poor)
  🥉 Recall: 64.4% (fair)
  😡 F1 Score: 43.0% (poor)
  😡 Dice: 43.0% (poor)
  🥉 Individual Tree Iou: 36.0% (fair)
  😡 Group Tree Iou: 12.6% (poor)

=== Concerns: ===
  • Low iou (24.3%) indicates poor iou
  • Low accuracy (59.3%) indicates poor accuracy
  • Low precision (33.8%) indicates poor precision
  • Low f1_score (43.0%) indicates poor f1 score
  • Low dice (43.0%) indicates poor dice
  • Low group_tree_iou (12.6%) indicates poor group tree iou

=== Recommendations: ===
  • Try a more powerful model (UNet, YOLOv8)
  • Reduce learning rate (try 1e-4 or 1e-5)
  • Increase training epochs
  • Check data quality and annotations
  • Consider data augmentation
